# M5 — Capstone: The 5-Step Playbook (UFC/BJJ edition)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/paiml/big-o-python-to-rust/blob/main/notebooks/m5-capstone.ipynb)

Course lesson 5.1.1 — the five-step playbook: **depyler transpile -> inspect & idiomatize -> author contract YAML -> criterion bench -> proptest property**. We apply it to a real problem: top-K hot fighters by win streak, solved three different ways. All three implementations must agree on the count multiset; only the asymptotic cost differs. Closing markdown beats lesson 5.2.1 (when NOT to translate).

## Step 1-2 — depyler transpile + idiomatize

Imagine running `depyler transpile top_k_hot.py`. The output is correct Rust but
often heuristic — uses `Vec` where `HashMap` would be faster, picks `clone()`
where `&str` would do. Step 2 is the human pass: replace heuristics with
idiomatic choices guided by what we know about the data shape.

Below: three Python implementations of "top-K hot fighters". Step 1 produced
each; step 2 was the choice between them.

In [1]:
import heapq


def top_k_naive(fighters: list[tuple[str, int]], k: int) -> list[tuple[str, int]]:
    """O(n^2) - for each candidate, scan for the max."""
    out: list[tuple[str, int]] = []
    remaining = list(fighters)
    while remaining and len(out) < k:
        best_idx = 0
        for i in range(1, len(remaining)):
            if remaining[i][1] > remaining[best_idx][1]:
                best_idx = i
        out.append(remaining.pop(best_idx))
    return out


def top_k_sort(fighters: list[tuple[str, int]], k: int) -> list[tuple[str, int]]:
    """O(n log n) — sort then take first k."""
    return sorted(fighters, key=lambda kv: -kv[1])[:k]


def top_k_heap(fighters: list[tuple[str, int]], k: int) -> list[tuple[str, int]]:
    """O(n log k) — min-heap of size k."""
    heap: list[tuple[int, str]] = []
    for name, streak in fighters:
        if len(heap) < k:
            heapq.heappush(heap, (streak, name))
        elif streak > heap[0][0]:
            heapq.heapreplace(heap, (streak, name))
    return sorted([(n, s) for s, n in heap], key=lambda kv: -kv[1])


fighters = [
    ("Khabib", 29),
    ("Silva", 16),
    ("Jones", 27),
    ("GSP", 13),
    ("Adesanya", 6),
    ("McGregor", 2),
    ("Diaz", 1),
    ("Volkanovski", 19),
    ("Cormier", 4),
    ("Holloway", 7),
]

top_naive = top_k_naive(fighters, 3)
top_sort = top_k_sort(fighters, 3)
top_heap = top_k_heap(fighters, 3)
print("top-3 naive:", top_naive)
print("top-3 sort :", top_sort)
print("top-3 heap :", top_heap)

top-3 naive: [('Khabib', 29), ('Jones', 27), ('Volkanovski', 19)]
top-3 sort : [('Khabib', 29), ('Jones', 27), ('Volkanovski', 19)]
top-3 heap : [('Khabib', 29), ('Jones', 27), ('Volkanovski', 19)]


## Step 3-5 — contract YAML + criterion bench + proptest property

**Step 3** — author the contract YAML. For top-K hot fighters, the relevant
contracts are `complexity-quadratic-v1` (naive), `complexity-linearithmic-v1`
(sort), and `complexity-linear-v1` (heap; technically O(n log k) but for fixed k
the dominant term is linear).

**Step 4** — `cargo bench` (criterion). The empirical receipt: confirms the
ratio bounds at the bench wall-clock level.

**Step 5** — proptest property: all three implementations agree on the same
multiset for every random roster.

In [2]:
def multiset(xs: list[tuple[str, int]]) -> frozenset[tuple[str, int]]:
    return frozenset(xs)


# The proptest-style property: all three impls produce the same top-K count multiset
assert multiset(top_naive) == multiset(top_sort)
assert multiset(top_sort) == multiset(top_heap)
assert len(top_naive) == 3
# Top fighter must be the actual hottest streak
assert max(s for _, s in top_naive) == 29
print("proptest: count-multiset equality naive == sort == heap")

proptest: count-multiset equality naive == sort == heap


## Lesson 5.2.1 — when NOT to translate

Not every Python program belongs in Rust. Three signals tell you to keep Python:

1. **The hot path is already C-accelerated.** numpy, pandas, scikit-learn — the
   inner loop is already native.  Translating the orchestration won't move the
   needle.
2. **The bottleneck is I/O, not CPU.** A web scraper waiting on HTTP doesn't get
   faster in Rust.
3. **The team owns Python and ships weekly.** Translation is a 1-2 quarter
   investment. Spend it on bottlenecks, not on convenience.

The playbook's first step is honest measurement. If `cProfile` shows the
bottleneck is in numpy, depyler can't help you. The same five steps apply once
you've identified a real CPU-bound bottleneck.

In [3]:
def should_translate(
    c_accelerated: bool, io_bound: bool, owns_python_only: bool
) -> bool:
    """Heuristic: only translate when none of the three skip signals fires."""
    return not (c_accelerated or io_bound or owns_python_only)


# Bottleneck is in numpy: skip translation
assert (
    should_translate(c_accelerated=True, io_bound=False, owns_python_only=False)
    is False
)
# Web scraper waiting on HTTP: skip translation
assert (
    should_translate(c_accelerated=False, io_bound=True, owns_python_only=False)
    is False
)
# Real CPU bottleneck and team has Rust capacity: translate
assert (
    should_translate(c_accelerated=False, io_bound=False, owns_python_only=False)
    is True
)
print(
    "when-not-to: only translate when CPU-bound + non-trivial constants + Rust capacity exists"
)

when-not-to: only translate when CPU-bound + non-trivial constants + Rust capacity exists


---
**Rust port:** [`m5-capstone/src/lib.rs`](../m5-capstone/src/lib.rs) ships all three top-K impls with proptest equivalence over random rosters + criterion benches for the wall-clock receipt. The Rust crate emits `contract: m5-capstone holds — OK` when the playbook completes. Course lessons 5.1.1, 5.2.1.